# 27. 複数の公開モデルをSRAで束ねる ― メリットは本当にあるか

GemmaやLlama、Qwenのような**公開済みの小型LLMを複数組み合わせる**と、本当に得をするのでしょうか?
このノートブックでは、SRAのルーティング思想を使って次の2つを検証します。

1. **A: ルーティング(専門性の使い分け)** — 入力ごとに得意なモデルへ振り分けると、混在ワークロードで最良の単体モデルを上回るか?
2. **B: アンサンブル(多数決)** — 同じ問題を複数モデルに解かせて多数決すると、コストに見合うほど精度が上がるか?

## モデル差異の「いちばん簡単な」吸収法
異なるモデルは **隠れ次元・トークナイザ・隠れ状態の意味空間** がすべて違うため、隠れ状態を素朴に結合すると壊れます。
そこで本ノートでは **各モデルを「テキスト入力→テキスト出力」のブラックボックス**として扱い、
ルーターは中立な共通埋め込み(SentenceTransformer)で判定します。各モデルの差異は薄いラッパー(自前のtokenizer+chat template)に封じ込め、**差異に触れない**のが要点です。
隠れ状態を本当に統合するアダプタ方式は、より高度な将来回に回します。

> **ライセンス注記**: Llama は Llama Community License、Gemma は Gemma Terms of Use の下で提供されます。利用条件を確認のうえ使用してください。Qwen は Apache-2.0 です。

In [ ]:
# === セットアップ(Colab T4想定) ===
# Colabには torch/transformers は同梱。以下は追加で必要。
!pip install -q sentence-transformers datasets bitsandbytes accelerate

import re, os, subprocess, tempfile
import numpy as np
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# Llama / Gemma は gated。利用申請済みのHFトークンでログインしてください:
# from huggingface_hub import login; login()  # ← トークンを入力
# gating を避けたい場合は、後段の 'general' モデルを非gatedの Qwen2.5-3B-Instruct に差し替え可。

In [ ]:
# === パラメータ化したLLMシナプス(差異吸収の核) ===
# model_id と specialty を引数化。各モデルが自分の tokenizer / chat template に閉じる。
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

class LLMSynapse:
    def __init__(self, model_id, specialty, load_4bit=True):
        self.model_id, self.specialty = model_id, specialty  # specialty=ルーティング用の得意分野記述
        self.tok = AutoTokenizer.from_pretrained(model_id)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        qcfg = BitsAndBytesConfig(load_in_4bit=True) if load_4bit else None
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=qcfg, device_map='auto', torch_dtype=torch.float16)
        self.model.eval()

    @torch.no_grad()
    def generate(self, user_text, max_new_tokens=512):
        msgs = [{'role': 'user', 'content': user_text}]
        prompt = self.tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        enc = self.tok(prompt, return_tensors='pt').to(self.model.device)
        out = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=self.tok.eos_token_id)
        return self.tok.decode(out[0, enc['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

In [ ]:
# === ドメイン別・採点器(出所非依存。correct と parse_ok を分離記録) ===
INSTRUCTIONS = {
    'code': 'Return ONLY the function `{entry_point}` inside a single ```python code block. No explanation.',
    'math': 'Solve step by step, then give the final answer as \\boxed{{number}}.',
    'qa':   'Answer with the single letter (A, B, C, or D) only.',
}

def build_prompt(item):
    instr = INSTRUCTIONS[item['domain']]
    if item['domain'] == 'code':
        instr = instr.format(entry_point=item['entry_point'])
    return f"{instr}\n\n{item['question']}"

def extract_code(resp):
    blocks = re.findall(r'```(?:python)?\s*\n(.*?)```', resp, re.DOTALL)
    return (blocks[-1] if blocks else resp).strip()

def grade_code(resp, item, timeout=8):
    code = extract_code(resp)
    if not code:
        return {'correct': False, 'parse_ok': False}
    program = f"{code}\n{item['test']}\ncheck({item['entry_point']})\n"
    path = None
    try:
        with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
            f.write(program); path = f.name
        proc = subprocess.run(['python3', '-I', path], capture_output=True, text=True, timeout=timeout)
        return {'correct': proc.returncode == 0, 'parse_ok': True}
    except subprocess.TimeoutExpired:
        return {'correct': False, 'parse_ok': True}
    finally:
        if path: os.unlink(path)

def extract_number(resp):
    m = re.search(r'\\boxed\{([^}]*)\}', resp)
    cand = m.group(1) if m else None
    if cand is None:
        nums = re.findall(r'-?\d[\d,]*\.?\d*', resp)
        if not nums: return None
        cand = nums[-1]
    try: return float(cand.replace(',', '').rstrip('.'))
    except ValueError: return None

def grade_math(resp, item, tol=1e-6):
    pred = extract_number(resp)
    if pred is None: return {'correct': False, 'parse_ok': False}
    return {'correct': abs(pred - float(item['answer'])) < tol, 'parse_ok': True}

def extract_choice(resp, options):
    m = re.search(r'answer\s*(?:is|:)?\s*\(?([A-D])\)?', resp, re.IGNORECASE)
    if m: return m.group(1).upper()
    m = re.search(r'\(?([A-D])[\).:]', resp)
    if m: return m.group(1).upper()
    hits = [k for k, v in options.items() if v and v.lower() in resp.lower()]
    if len(hits) == 1: return hits[0]
    letters = re.findall(r'\b([A-D])\b', resp)
    if letters: return letters[-1].upper()
    return None

def grade_qa(resp, item):
    pred = extract_choice(resp, item['options'])
    if pred is None: return {'correct': False, 'parse_ok': False}
    return {'correct': pred == item['answer'], 'parse_ok': True}

GRADERS = {'code': grade_code, 'math': grade_math, 'qa': grade_qa}

def grade(item, resp):
    r = GRADERS[item['domain']](resp, item); r['domain'] = item['domain']; return r

In [ ]:
# === ゼロショット意味ルーター(ルーティング空間をモデル内部から切り離す) ===
from sentence_transformers import SentenceTransformer

_st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def encode(texts):
    return _st.encode(list(texts), normalize_embeddings=True)

class SemanticRouter:
    def __init__(self, specialties, encoder):
        self.names = list(specialties)
        self.spec_vecs = np.asarray(encoder([specialties[n] for n in self.names]))
        self.encoder = encoder
    def route(self, query):
        q = np.asarray(self.encoder([query])[0])
        sims = self.spec_vecs @ q
        best = int(sims.argmax())
        return self.names[best], dict(zip(self.names, sims.tolist()))

In [ ]:
# === 評価データ: 混在ワークロード(code + math + qa) ===
# ベンチサブセット(HumanEval / GSM8K / MMLU)を共通スキーマへ写像。採点器は変更不要。
from datasets import load_dataset

def load_humaneval(n=10):
    ds = load_dataset('openai/openai_humaneval', split='test')
    return [{'domain': 'code', 'question': r['prompt'], 'entry_point': r['entry_point'], 'test': r['test']}
            for r in ds.select(range(min(n, len(ds))))]

def load_gsm8k(n=10):
    ds = load_dataset('openai/gsm8k', 'main', split='test')
    out = []
    for r in ds.select(range(min(n, len(ds)))):
        gold = r['answer'].split('####')[-1].replace(',', '').strip()
        out.append({'domain': 'math', 'question': r['question'], 'answer': float(gold)})
    return out

def load_mmlu(n=10, subject='high_school_geography'):
    L = 'ABCD'
    ds = load_dataset('cais/mmlu', subject, split='test')
    out = []
    for r in ds.select(range(min(n, len(ds)))):
        opts = {L[i]: c for i, c in enumerate(r['choices'])}
        q = r['question'] + '\n' + ' '.join(f'({L[i]}) {c}' for i, c in enumerate(r['choices']))
        out.append({'domain': 'qa', 'question': q, 'options': opts, 'answer': L[r['answer']]})
    return out

N = 10  # 各ドメイン件数(T4の時間予算に応じて調整)
items = load_humaneval(N) + load_gsm8k(N) + load_mmlu(N)
print('total items:', len(items))

In [ ]:
# === 3つの公開モデルをシナプスとして登録(別ファミリー混在) ===
# 専門家2 + 汎用1。Qwenは非gated、Llamaはgated(要HFログイン)。
synapses = {
    'coder':   LLMSynapse('Qwen/Qwen2.5-Coder-1.5B-Instruct', 'writing and fixing program code, Python functions, algorithms'),
    'math':    LLMSynapse('Qwen/Qwen2.5-Math-1.5B-Instruct',  'solving arithmetic and math word problems, numeric reasoning'),
    'general': LLMSynapse('microsoft/Phi-3.5-mini-instruct', 'general knowledge questions, facts, history, geography'),
    # 非gated・別ファミリー。代替: ibm-granite/granite-3.1-2b-instruct(軽量) / 同族で良ければ Qwen/Qwen2.5-3B-Instruct
}
specialties = {name: s.specialty for name, s in synapses.items()}
router = SemanticRouter(specialties, encode)

# オラクル用: ドメイン→専門シナプスの対応(理論上限の振り分け)
DOMAIN_TO_SYNAPSE = {'code': 'coder', 'math': 'math', 'qa': 'general'}

In [ ]:
# === 全モデル×全問題の応答を一度だけ生成(=アンサンブルのコスト基準) ===
# これを行列として持ち、各実験はここから読み出す。
from collections import defaultdict
import time

responses = defaultdict(dict)  # responses[model_name][i] = 応答テキスト
t0 = time.time()
for name, syn in synapses.items():
    for i, item in enumerate(items):
        prompt = build_prompt(item)  # ドメインに応じた答え方の指示を付与(全モデル同条件)
        responses[name][i] = syn.generate(prompt)
    print(f'{name}: done ({time.time()-t0:.0f}s)')
print('total generations:', len(synapses) * len(items))

In [ ]:
# === ベースライン: 各モデル単体 × 各ドメイン精度 ===
# 「全部得意な単体はいない」ことを可視化。
import pandas as pd

domains = ['code', 'math', 'qa']
rows = {}
for name in synapses:
    row = {}
    for d in domains:
        idxs = [i for i, it in enumerate(items) if it['domain'] == d]
        acc = np.mean([grade(items[i], responses[name][i])['correct'] for i in idxs])
        row[d] = round(float(acc), 3)
    idxs_all = range(len(items))
    row['mixed(all)'] = round(float(np.mean([grade(items[i], responses[name][i])['correct'] for i in idxs_all])), 3)
    rows[name] = row
baseline_df = pd.DataFrame(rows).T
print(baseline_df)
single_best = baseline_df['mixed(all)'].idxmax()
print('\n最良の単体モデル(混在全体):', single_best, '=', baseline_df.loc[single_best, 'mixed(all)'])

In [ ]:
# === 主結果★ A: 混在セットで 単体ベスト / オラクル / SRAルーティング を比較 ===
def acc_and_calls(pick_fn):
    correct, parse_ok = [], []
    for i, item in enumerate(items):
        name = pick_fn(item)
        r = grade(item, responses[name][i])
        correct.append(r['correct']); parse_ok.append(r['parse_ok'])
    return float(np.mean(correct)), float(np.mean(parse_ok)), len(items)  # calls = N

# 単体ベスト: 常に同じモデル
acc_single, _, calls_single = acc_and_calls(lambda it: single_best)
# オラクル(上限): ドメインの専門家へ必ず振る
acc_oracle, _, calls_oracle = acc_and_calls(lambda it: DOMAIN_TO_SYNAPSE[it['domain']])
# SRAルーティング: ルーターが質問から選択
acc_sra, parse_sra, calls_sra = acc_and_calls(lambda it: router.route(it['question'])[0])

print(f'{"single-best ("+single_best+")":28s} acc={acc_single:.3f}  calls={calls_single}')
print(f'{"oracle routing (upper bound)":28s} acc={acc_oracle:.3f}  calls={calls_oracle}')
print(f'{"SRA semantic routing":28s} acc={acc_sra:.3f}  calls={calls_sra}  (router parse_ok={parse_sra:.2f})')

# ルーター混同行列: 各ドメインがどのシナプスへ流れたか
conf = defaultdict(lambda: defaultdict(int))
for it in items:
    conf[it['domain']][router.route(it['question'])[0]] += 1
print('\nルーティング先 (行=ドメイン, 列=選ばれたシナプス):')
print(pd.DataFrame(conf).T.fillna(0).astype(int))

In [ ]:
# === 対照実験 B: 多数決アンサンブル(コストに見合うか) ===
# 多数決は『抽出された答えが比較可能』なとき定義できる。
# → math(数値)/ qa(文字)で実施。code は自由生成のため投票が定義困難なので対象外(正直なスコープ)。
from collections import Counter

def extracted_answer(item, resp):
    if item['domain'] == 'math': return extract_number(resp)
    if item['domain'] == 'qa':   return extract_choice(resp, item['options'])
    return None

votable = [i for i, it in enumerate(items) if it['domain'] in ('math', 'qa')]

def ensemble_correct(i):
    item = items[i]
    preds = [extracted_answer(item, responses[n][i]) for n in synapses]
    preds = [p for p in preds if p is not None]
    if not preds: return False
    winner = Counter(map(str, preds)).most_common(1)[0][0]  # 多数決
    # 勝者を該当ドメインの正解と照合
    if item['domain'] == 'math':
        try: return abs(float(winner) - float(item['answer'])) < 1e-6
        except ValueError: return False
    return winner == item['answer']

acc_ens = float(np.mean([ensemble_correct(i) for i in votable]))
# 同じ可投票サブセットでの SRA と 単体ベスト(公平比較)
acc_sra_sub = float(np.mean([grade(items[i], responses[router.route(items[i]['question'])[0]][i])['correct'] for i in votable]))
acc_single_sub = float(np.mean([grade(items[i], responses[single_best][i])['correct'] for i in votable]))

print('=== math+qa サブセットでの比較 ===')
print(f'single-best       acc={acc_single_sub:.3f}  calls={len(votable)}')
print(f'SRA routing       acc={acc_sra_sub:.3f}  calls={len(votable)}')
print(f'majority ensemble acc={acc_ens:.3f}  calls={len(votable)*len(synapses)}  (=3倍のコスト)')

## 結論(正直版)

- **A(ルーティング)**: 混在ワークロードでは、得意なモデルへ振り分けるだけで **最良の単体モデルを上回り**、オラクル上限に近づく。
  これは「全部入りの巨大1モデル」を、安価な小型専門家の集合で代替できる可能性を示す。
- **B(アンサンブル)**: 同じ問題を全モデルに解かせる多数決は **コスト3倍の割に精度の伸びは限定的**。
  公開モデルは重なったWebデータで学習しており誤りが相関しやすいため、アンサンブル利得は小さくなりがち。
- **差異吸収**: 隠れ状態に触れず **テキスト境界で束ねる**のが最も簡単で堅牢。次元・トークナイザ・空間の非互換を回避できる。

> **注意**: 結果は `N`、モデル選定、MMLU科目、デコード設定に依存する。`router parse_ok` や各ドメインの結果を必ず併記し、
> 「モデルが弱い」のか「採点器が読めていない」のかを区別すること。

## 次の一歩
- **隠れ状態の統合(アダプタ方式)**: テキスト境界ではなく、各モデルの隠れ状態を射影アダプタでSRA共通空間にアライメントする上級回。
- **Gemma + Llama + Qwen の3ファミリー版**: 本ノートの枠組みをそのまま3ファミリーへ拡張する。